# Week 5 Lab 03: Cordwell Support Assistant
## A RAG pipeline with a fine-tuned adapter, evaluated end to end

Cordwell Home & Hardware is rolling out a documentation assistant for its
support team. Module 01 gave you a LoRA adapter trained to answer in strict
JSON. Module 02 gave you a vector index of the product documentation. This
lab wires them into one retrieval augmented generation chain and then does
what a production team must do before shipping: measure it.

**Objectives.** By the end of this lab you can:

1. Build a retrieval chain with `create_retrieval_chain` and
   `create_stuff_documents_chain`, including a document prompt that makes
   citations possible.
2. Implement format adherence and abstention checks for structured model
   output.
3. Implement faithfulness (claim decomposition) and answer relevancy
   metrics with an injected judge model.
4. Run a base versus adapter comparison over an evaluation set and read
   the results.
5. Demonstrate the abstention trade off: why a fine-tune that always
   answers is a liability, not a feature.

**Time budget (about 3.5 hours core, 30 min stretch).**

| Part | Focus | Time |
|---|---|---|
| A | Load corpus, retriever, model (given) | 20 min |
| B | Build the RAG chain | 35 min |
| C | Baseline run, format and abstention checks | 40 min |
| D | Faithfulness and relevancy | 60 min |
| E | Full evaluation, base versus adapter | 30 min |
| F | Abstention trade off on unanswerable questions | 25 min |
| G | Stretch: degrade retrieval, diagnose | 30 min |

Run every cell in order. Checks print PASS, FAIL, or TODO and never crash
the notebook. There are 22 checks; a fresh notebook starts at 3.


### Backends

The lab runs in two modes, selected by environment variable before you
start Jupyter. **Offline is the default and is what the checks are
calibrated against.** Local mode uses the real model and index and is
covered in `setup/LOCAL_MODE_SETUP.md`; its outputs vary slightly run to
run, so treat check thresholds there as guidance.

| Variable | Values | Default |
|---|---|---|
| `RAG_BACKEND` | `offline`, `local` | `offline` |
| `JUDGE_BACKEND` | `inprocess`, `lmstudio`, `ollama` | `inprocess` |
| `JUDGE_MODEL` | any served model tag | `gemma4` |

The two judge servers are interchangeable: LM Studio serves an OpenAI
compatible endpoint on port 1234 and Ollama serves one on port 11434, so
one client code path covers both. Confirm the exact model tag your
machine has pulled before class.


In [ ]:
%pip install -r requirements.txt

In [ ]:
import json
import os
import re
from pathlib import Path

import pandas as pd
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate

# Currency note: on langchain 1.x these two helpers live in the
# langchain-classic package. from langchain.chains import ... no longer works.
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from cordwell_rag_backend import (
    ABSTAIN_TEXT, CLAIM_PROMPT, RELEVANCY_PROMPT, VERIFY_PROMPT,
    build_backend, docs_to_context, load_corpus, make_judge,
)

# ---- configuration (models and endpoints live here, never inline) ----
RAG_BACKEND = os.environ.get("RAG_BACKEND", "local")
JUDGE_BACKEND = os.environ.get("JUDGE_BACKEND", "inprocess")
JUDGE_MODEL = os.environ.get("JUDGE_MODEL", "gemma4")

DATA_DIR = Path("data")
CORPUS_PATH = DATA_DIR / "cordwell_corpus.jsonl"
QUERIES_PATH = DATA_DIR / "eval_queries.json"
TOP_K = 4

# Local mode only (ignored offline). See setup/LOCAL_MODE_SETUP.md.
BASE_MODEL = str(Path(os.environ.get("BASE_MODEL", 
                                     "~/models/SmolLM2-360M-Instruct")).expanduser().resolve())
EMBED_MODEL = str(Path(os.environ.get("EMBED_MODEL",
                                     "~/models/all-MiniLM-L6-v2")).expanduser().resolve())
ADAPTER_PATH = os.environ.get("ADAPTER_PATH", "./adapters/cordwell")
PINECONE_HOST = os.environ.get("PINECONE_HOST", "http://localhost:5080")
INDEX_NAME = os.environ.get("INDEX_NAME", "cordwell-support")

local_cfg = dict(base_model=BASE_MODEL, adapter_path=ADAPTER_PATH,
                 pinecone_host=PINECONE_HOST, index_name=INDEX_NAME, 
                 embed_model=EMBED_MODEL)
BACKEND = build_backend(RAG_BACKEND, CORPUS_PATH, k=TOP_K,
                        **(local_cfg if RAG_BACKEND == "local" else {}))
RETRIEVER = BACKEND.retriever
LLM = BACKEND.llm
set_adapter = BACKEND.set_adapter
JUDGE = make_judge(JUDGE_BACKEND, BACKEND, model=JUDGE_MODEL)

QUERIES = json.load(open(QUERIES_PATH))
ANSWERABLE = QUERIES["answerable"]
UNANSWERABLE = QUERIES["unanswerable"]
CORPUS = load_corpus(CORPUS_PATH)

print(f"backend={BACKEND.name}  judge={JUDGE_BACKEND}  top_k={TOP_K}")
print(f"corpus chunks: {len(CORPUS)}  answerable: {len(ANSWERABLE)}  unanswerable: {len(UNANSWERABLE)}")

In [ ]:
# Soft check harness: checks report, they never crash the notebook.
RESULTS = {}

def soft(name, fn):
    try:
        ok = bool(fn())
        RESULTS[name] = ok
        print(("PASS  " if ok else "FAIL  ") + name)
    except NotImplementedError:
        RESULTS[name] = False
        print("TODO  " + name + "  (function not implemented yet)")
    except Exception as e:
        RESULTS[name] = False
        print(f"FAIL  {name}  ({type(e).__name__}: {e})")

def score():
    total = len(RESULTS)
    passed = sum(RESULTS.values())
    print(f"{passed}/{total} checks passing")

## Part A. Load and inspect the pieces (given)

Everything in this part is provided; your job is to run it and read the
output. Three artifacts come together in this lab:

* **The corpus.** 14 chunks of Cordwell product documentation in
  `data/cordwell_corpus.jsonl`, each with a `source` id like
  `install_manual_th40.md#7`. In local mode these live in the Pinecone
  index instead; the ids are identical.
* **The retriever.** Offline, a deterministic keyword scorer; local, the
  Module 02 vector index. Either way it answers `.invoke(query)` with a
  list of `Document` objects, which is all the chain cares about.
* **The model.** Offline, a scripted stand-in that reproduces the exact
  behavior shapes of the real pair from Module 01: the base model answers
  in chatty prose, the adapter answers in strict JSON. `set_adapter(True)`
  and `set_adapter(False)` switch between them.

Look at what the retriever returns for the probe query below. Notice the
top hit for the circuit breaker question is the quickstart chunk, which
*mentions* the breaker panel without giving the sizing answer. Remember
that chunk; it matters again in Part G.


In [ ]:
probe_query = "What size circuit breaker does the TH-40 thermostat need?"
probe_docs = RETRIEVER.invoke(probe_query)
for d in probe_docs:
    print(f"[{d.metadata['source']}]")
    print("   ", d.page_content[:100].strip(), "...")
    print()

soft("A1_corpus_loaded", lambda: len(CORPUS) == 14 and
     all("source" in d.metadata for d in CORPUS))
soft("A2_retriever_probe", lambda: len(probe_docs) == TOP_K and
     all(isinstance(d, Document) for d in probe_docs))
soft("A3_backend_ready", lambda: LLM is not None and callable(JUDGE)
     and callable(set_adapter))

## Part B. Build the RAG chain (35 min)

The modern LangChain retrieval pattern is two helpers composed together:

* `create_stuff_documents_chain(llm, prompt, document_prompt=...)` takes
  retrieved documents, renders each one through `document_prompt`, stuffs
  them into the `{context}` slot of your chat prompt, and calls the model.
* `create_retrieval_chain(retriever, combine_docs_chain)` runs the
  retriever first and feeds its output to the stuff chain.

Three contract details trip people up, so here they are up front:

1. Your chat prompt **must** contain a `{context}` variable, or the
   stuff chain has nowhere to put the documents.
2. The chain's input key is `input`, not `query` or `question`:
   `chain.invoke({"input": "..."})`.
3. The result is a dict with keys `input`, `context` (the retrieved
   `Document` list), and `answer` (the model text).

One more, specific to this lab: the system prompt tells the model to cite
sources, but the model can only cite what it can see. The default
document rendering is bare page content with **no source id**. Pass a
`document_prompt` that renders each document as `[source] content` so the
ids are in front of the model.

**Worked target output.** With the adapter enabled, your finished chain
on the probe query returns exactly this (offline backend):

```python
result = CHAIN.invoke({"input": "What size circuit breaker does the TH-40 thermostat need?"})
sorted(result.keys())   # ['answer', 'context', 'input']
len(result["context"])  # 4
result["answer"]
# {"answer": "The TH-40 requires a 15 amp circuit breaker.",
#  "citations": ["install_manual_th40.md#7"], "confidence": "high"}
```

**Done when** checks B1 through B3 pass and the hand query loop prints a
JSON answer with a citation for all five questions.


In [ ]:
SYSTEM = (
    "You are the Cordwell Home & Hardware support assistant. Answer the "
    "customer's question using only the provided context. Respond with a JSON "
    "object with keys answer, citations, and confidence. If the context does "
    "not contain the answer, say so instead of guessing."
)
print(SYSTEM)

In [ ]:
# ============ SOLUTION ============
def build_chain(llm, retriever):
    """Build the Cordwell RAG chain. See the contract in the lab text."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM + "\n\nContext:\n{context}"),
        ("human", "{input}"),
    ])
    doc_prompt = PromptTemplate.from_template("[{source}] {page_content}")
    combine_docs = create_stuff_documents_chain(llm, prompt, document_prompt=doc_prompt)
    return create_retrieval_chain(retriever, combine_docs)

In [ ]:
CHAIN = None
RESULT = None
set_adapter(True)
try:
    CHAIN = build_chain(LLM, RETRIEVER)
    RESULT = CHAIN.invoke({"input": probe_query})
    print("answer: ", RESULT["answer"])
    print("sources:", [d.metadata["source"] for d in RESULT["context"]])
except NotImplementedError:
    print("build_chain not implemented yet. Complete the TODO above, then re-run this cell.")

soft("B1_chain_built", lambda: CHAIN is not None and hasattr(CHAIN, "invoke"))
soft("B2_result_keys", lambda: RESULT is not None and
     {"input", "context", "answer"} <= set(RESULT))
soft("B3_context_documents", lambda: RESULT is not None and
     len(RESULT["context"]) == TOP_K and
     all(isinstance(d, Document) for d in RESULT["context"]))

In [ ]:
# Hand queries: five real support questions through your chain, adapter on.
HAND_QUERIES = [
    "What size circuit breaker does the TH-40 thermostat need?",
    "What color temperature is the SL-30 shop light?",
    "How many cycles per day is the GD-200 rated for?",
    "How do I pair the TH-40 with wifi?",
    "What is the return window for online orders?",
]
if CHAIN is None:
    print("Complete Part B first.")
else:
    set_adapter(True)
    for q in HAND_QUERIES:
        r = CHAIN.invoke({"input": q})
        print(f"Q: {q}\nA: {r['answer']}\n")

## Part C. Baseline and the format metric (40 min)

Before measuring the adapter, measure what you paid to fix. The base
model was never trained to emit JSON, and the support platform that
consumes these answers is a JSON parser. **Format adherence** is
therefore the first metric: does the answer parse, and does it match the
schema exactly?

`format_check(answer)` returns a tuple `(ok, parsed)`:

* `ok` is True only if the whole string parses as JSON, is an object,
  has **exactly** the keys `answer`, `citations`, `confidence` (no
  extras, none missing), `answer` is a string, `citations` is a list,
  and `confidence` is one of `high`, `medium`, `low`.
* `parsed` is the dict when `ok` is True, otherwise None.

Strictness is the point: an answer wrapped in prose like "Sure! Here is
the JSON..." fails, because the parser downstream would fail.

`abstention_check(answer)` returns True when the model declined to
answer, in either dialect it might use:

* a valid JSON answer with `confidence` equal to `low` **and** empty
  `citations`, or
* prose matching refusal phrasing such as "could not find",
  "don't have that information", "no information about", "not mentioned
  in", or "unable to find" (case insensitive).

**Worked target output.**

```python
format_check('{"answer": "The TH-40 requires a 15 amp circuit breaker.", "citations": ["install_manual_th40.md#7"], "confidence": "high"}')
# (True, {'answer': 'The TH-40 requires a 15 amp circuit breaker.',
#         'citations': ['install_manual_th40.md#7'], 'confidence': 'high'})

format_check('Sure! Here is the JSON: {"answer": "x", "citations": [], "confidence": "high"}')
# (False, None)

format_check('{"answer": "x", "citations": [], "confidence": "definitely"}')
# (False, None)

abstention_check('{"answer": "I could not find that information in the Cordwell documentation.", "citations": [], "confidence": "low"}')
# True
```

**Done when** C1 through C6 pass. C6 runs the baseline: the base model
should produce zero format compliant answers on the six answerable
questions while abstaining on none of them, which is the before picture
the adapter has to beat.


In [ ]:
# ============ SOLUTION ============
def format_check(answer):
    """Strict schema check for a support answer. Returns (ok, parsed)."""
    try:
        obj = json.loads(answer.strip())
    except json.JSONDecodeError:
        return False, None
    if not isinstance(obj, dict):
        return False, None
    if set(obj) != {"answer", "citations", "confidence"}:
        return False, None
    if not isinstance(obj["answer"], str) or not isinstance(obj["citations"], list):
        return False, None
    if obj["confidence"] not in {"high", "medium", "low"}:
        return False, None
    return True, obj

In [ ]:
# ============ SOLUTION ============
ABSTAIN_PATTERNS = [
    r"could not find", r"couldn'?t find", r"don'?t have (that|any)? ?information",
    r"no information about", r"not (mentioned|covered) in", r"unable to find",
]

def abstention_check(answer):
    """Detect a declined answer in either dialect. Returns a bool."""
    ok, obj = format_check(answer)
    text = obj["answer"] if ok else answer
    if ok and obj["confidence"] == "low" and not obj["citations"]:
        return True
    return any(re.search(p, text.lower()) for p in ABSTAIN_PATTERNS)

In [ ]:
VALID_JSON = ('{"answer": "The TH-40 requires a 15 amp circuit breaker.", '
              '"citations": ["install_manual_th40.md#7"], "confidence": "high"}')
PREAMBLE = ('Sure! Here is the JSON: {"answer": "x", "citations": [], '
            '"confidence": "high"}')
BAD_SCHEMA = '{"answer": "x", "citations": [], "confidence": "definitely"}'
MISSING_KEY = '{"answer": "x", "confidence": "high"}'
ABSTAIN_JSON = ('{"answer": "I could not find that information in the Cordwell '
                'documentation.", "citations": [], "confidence": "low"}')

soft("C1_format_valid_accepted", lambda: format_check(VALID_JSON) ==
     (True, json.loads(VALID_JSON)))
soft("C2_format_preamble_rejected", lambda: format_check(PREAMBLE) == (False, None))
soft("C3_format_schema_rejected", lambda: format_check(BAD_SCHEMA) == (False, None)
     and format_check(MISSING_KEY) == (False, None))
soft("C4_abstention_detected", lambda: abstention_check(ABSTAIN_JSON) is True and
     abstention_check("I'm sorry, but I could not find that.") is True)
soft("C5_abstention_negative", lambda: abstention_check(VALID_JSON) is False)

In [ ]:
# Baseline: adapter OFF, all six answerable questions.
BASE_ROWS = []
if CHAIN is None:
    print("Complete Part B first.")
else:
    try:
        set_adapter(False)
        for q in ANSWERABLE:
            r = CHAIN.invoke({"input": q["question"]})
            fmt, _ = format_check(r["answer"])
            BASE_ROWS.append({"question": q["question"], "answer": r["answer"],
                              "format_valid": fmt,
                              "abstained": abstention_check(r["answer"])})
        for row in BASE_ROWS:
            print(f"format={row['format_valid']}  {row['answer'][:90]}...")
    except NotImplementedError:
        print("format_check or abstention_check not implemented yet.")
    finally:
        set_adapter(True)

soft("C6_baseline_format_rate", lambda: len(BASE_ROWS) == 6 and
     not any(r["format_valid"] for r in BASE_ROWS) and
     not any(r["abstained"] for r in BASE_ROWS))

## Part D. Faithfulness and relevancy (60 min)

Format tells you the answer is parseable. It says nothing about whether
the answer is *true to the retrieved context* or *on topic*. Those are
the next two metrics, and both use a **judge**: a second model call that
evaluates the first model's output. The judge arrives as an injected
callable `JUDGE` (a function from prompt string to reply string), so the
same metric code works whether the judge is the in-process base model, LM
Studio, or Ollama.

**Faithfulness** follows the claim decomposition recipe from the slides:

1. `decompose_claims(answer, judge)`: ask the judge to break the answer
   into atomic factual claims using `CLAIM_PROMPT`, one claim per line.
   Split the reply on newlines, strip each line, and drop empty lines.
2. `verify_claim(claim, context, judge)`: ask the judge with
   `VERIFY_PROMPT` whether the claim can be inferred from the context.
   The reply counts as supported when, after stripping and uppercasing,
   it starts with YES.
3. `faithfulness(answer, context, judge)`: decompose, verify each claim,
   and return supported claims divided by total claims. **Zero claims
   returns 0.0**: an answer that asserts nothing checkable earns no
   trust, a convention worth debating with your group later.

**Answer relevancy** is one judge call: `answer_relevancy(answer,
question, judge)` formats `RELEVANCY_PROMPT` and returns True when the
reply starts with YES after stripping and uppercasing.

The three prompts are already defined; run the cell below to read them.
Context strings come from `docs_to_context(docs)`, which renders
documents exactly the way the chain prompt does.

**Worked target output.** The base model's verbose probe answer,
decomposed and verified against the two fixture chunks:

```python
BASE_STYLE = ("Certainly! I'd be happy to help. Based on the documentation "
              "provided, the TH-40 requires a 15 amp circuit breaker. "
              "Let me know if you need anything else!")
decompose_claims(BASE_STYLE, JUDGE)
# ['The TH-40 requires a 15 amp circuit breaker.']
verify_claim('The TH-40 requires a 15 amp circuit breaker.', FIX_CONTEXT, JUDGE)
# True
verify_claim('The TH-40 requires a 40 amp circuit breaker.', FIX_CONTEXT, JUDGE)
# False
faithfulness(BASE_STYLE, FIX_CONTEXT, JUDGE)
# 1.0
```

Note what the decomposition did: the greeting and sign off vanished, and
"Based on the documentation provided" was stripped from the claim. Judges
are prompted to ignore pleasantries so faithfulness measures facts, not
padding.

**Done when** D1 through D6 pass.


In [ ]:
for name, p in [("CLAIM_PROMPT", CLAIM_PROMPT),
                ("VERIFY_PROMPT", VERIFY_PROMPT),
                ("RELEVANCY_PROMPT", RELEVANCY_PROMPT)]:
    print(f"----- {name} -----")
    print(p)
    print()

In [ ]:
# ============ SOLUTION ============
def decompose_claims(answer, judge):
    """Ask the judge to split an answer into atomic factual claims."""
    raw = judge(CLAIM_PROMPT.format(answer=answer))
    return [c.strip() for c in raw.split("\n") if c.strip()]

In [ ]:
# ============ SOLUTION ============
def verify_claim(claim, context, judge):
    """Ask the judge whether the context supports one claim."""
    out = judge(VERIFY_PROMPT.format(context=context, claim=claim))
    return out.strip().upper().startswith("YES")


def faithfulness(answer, context, judge):
    """Fraction of the answer's claims supported by the context."""
    claims = decompose_claims(answer, judge)
    if not claims:
        return 0.0
    supported = sum(1 for c in claims if verify_claim(c, context, judge))
    return supported / len(claims)

In [ ]:
# ============ SOLUTION ============
def answer_relevancy(answer, question, judge):
    """Ask the judge whether the answer addresses the question."""
    out = judge(RELEVANCY_PROMPT.format(question=question, answer=answer))
    return out.strip().upper().startswith("YES")

In [ ]:
FIX_DOCS = [d for d in CORPUS if d.metadata["id"] in {"th40-007", "returns-001"}]
FIX_CONTEXT = docs_to_context(FIX_DOCS)

BASE_STYLE = ("Certainly! I'd be happy to help. Based on the documentation "
              "provided, the TH-40 requires a 15 amp circuit breaker. "
              "Let me know if you need anything else!")
TWO_CLAIM = ("The TH-40 requires a 15 amp circuit breaker. "
             "The TH-40 ships with a 40 foot power cable.")
BREAKER_Q = "What size circuit breaker does the TH-40 thermostat need?"
GOOD_ANSWER = "The TH-40 requires a 15 amp circuit breaker."
OFF_TOPIC = "The TH-40 supports 2.4 GHz wifi networks only."

soft("D1_claims_decomposed", lambda: (lambda cl: len(cl) == 1 and
     "15 amp" in cl[0])(decompose_claims(BASE_STYLE, JUDGE)))
soft("D2_claim_supported", lambda: verify_claim(
     "The TH-40 requires a 15 amp circuit breaker.", FIX_CONTEXT, JUDGE) is True)
soft("D3_claim_fabricated", lambda: verify_claim(
     "The TH-40 requires a 40 amp circuit breaker.", FIX_CONTEXT, JUDGE) is False)
soft("D4_faithfulness_full", lambda: faithfulness(BASE_STYLE, FIX_CONTEXT, JUDGE) == 1.0)
soft("D5_faithfulness_partial", lambda: faithfulness(TWO_CLAIM, FIX_CONTEXT, JUDGE) == 0.5)
soft("D6_relevancy_pair", lambda: answer_relevancy(GOOD_ANSWER, BREAKER_Q, JUDGE)
     is True and answer_relevancy(OFF_TOPIC, BREAKER_Q, JUDGE) is False)

## Part E. The full evaluation loop (30 min)

Now wire your metrics into an evaluation harness and run the before and
after comparison properly. Two functions:

`evaluate_set(chain, queries, judge)` runs every query through the chain
and returns a `pandas.DataFrame` with one row per query and these
columns:

* `question`, `answer`: the query text and the raw model answer
* `sources`: list of source ids of the retrieved documents
* `format_valid`: your format_check verdict (the bool)
* `abstained`: your abstention_check verdict
* `faithfulness`: your faithfulness score against the retrieved context
  rendered with `docs_to_context`, **or None when the model abstained**
  (an abstention asserts nothing to verify)
* `relevant`: your answer_relevancy verdict
* `context_hit`: only when the query dict has a `gold_source` key, True
  when that source id appears in `sources` (this is context recall at
  the single gold chunk level)

`summarize(df)` reduces a results frame to a dict with keys
`format_rate` (mean of format_valid), `abstention_rate` (mean of
abstained), `faithfulness` (mean of the non-null faithfulness values,
None if every row abstained), and `relevancy_rate` (mean of relevant).
Return plain Python floats.

**Worked target output.** Adapter enabled, the six answerable questions:

```text
   format_valid  abstained  faithfulness  relevant  context_hit
0          True      False           1.0      True         True
1          True      False           1.0      True         True
2          True      False           1.0      True         True
3          True      False           1.0      True         True
4          True      False           1.0      True         True
5          True      False           1.0      True         True
```

```python
{"base": {"format_rate": 0.0, "abstention_rate": 0.0,
          "faithfulness": 1.0, "relevancy_rate": 1.0},
 "lora": {"format_rate": 1.0, "abstention_rate": 0.0,
          "faithfulness": 1.0, "relevancy_rate": 1.0}}
```

Read that comparison closely before moving on. The base model is
**already faithful and relevant**: it finds the right facts and states
them truthfully. The only thing the adapter bought on these questions is
format. Hold that thought; Part F shows what else the adapter changed.

**Done when** E1 and E2 pass.


In [ ]:
# ============ SOLUTION ============
def evaluate_set(chain, queries, judge):
    """Run an evaluation set through the chain and score every answer."""
    rows = []
    for q in queries:
        res = chain.invoke({"input": q["question"]})
        answer, docs = res["answer"], res["context"]
        context = docs_to_context(docs)
        fmt, _ = format_check(answer)
        abst = abstention_check(answer)
        row = {
            "question": q["question"],
            "answer": answer,
            "sources": [d.metadata["source"] for d in docs],
            "format_valid": fmt,
            "abstained": abst,
            "faithfulness": None if abst else faithfulness(answer, context, judge),
            "relevant": answer_relevancy(answer, q["question"], judge),
        }
        if "gold_source" in q:
            row["context_hit"] = q["gold_source"] in row["sources"]
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(df):
    """Reduce a results frame to one dict of rates."""
    faith = df["faithfulness"].dropna()
    return {
        "format_rate": float(df["format_valid"].mean()),
        "abstention_rate": float(df["abstained"].mean()),
        "faithfulness": float(faith.mean()) if len(faith) else None,
        "relevancy_rate": float(df["relevant"].mean()),
    }

In [ ]:
DF_BASE = DF_LORA = COMPARISON = None
SHOW_COLS = ["format_valid", "abstained", "faithfulness", "relevant", "context_hit"]
if CHAIN is None:
    print("Complete Part B first.")
else:
    try:
        set_adapter(False)
        DF_BASE = evaluate_set(CHAIN, ANSWERABLE, JUDGE)
        set_adapter(True)
        DF_LORA = evaluate_set(CHAIN, ANSWERABLE, JUDGE)
        print("=== base ===");  print(DF_BASE[SHOW_COLS].to_string())
        print("=== adapter ==="); print(DF_LORA[SHOW_COLS].to_string())
        COMPARISON = {"base": summarize(DF_BASE), "lora": summarize(DF_LORA)}
        print(json.dumps(COMPARISON, indent=2))
    except NotImplementedError:
        print("evaluate_set or summarize not implemented yet.")
    finally:
        set_adapter(True)

soft("E1_evaluate_set", lambda: DF_LORA is not None and len(DF_LORA) == 6 and
     {"question", "answer", "sources", "format_valid", "abstained",
      "faithfulness", "relevant", "context_hit"} <= set(DF_LORA.columns) and
     DF_LORA["format_valid"].all() and DF_LORA["context_hit"].all() and
     not DF_BASE["format_valid"].any())
soft("E2_summary_comparison", lambda: COMPARISON is not None and
     COMPARISON["lora"]["format_rate"] == 1.0 and
     COMPARISON["base"]["format_rate"] == 0.0 and
     COMPARISON["lora"]["faithfulness"] >= 0.99 and
     COMPARISON["base"]["abstention_rate"] == 0.0)

## Part F. The abstention trade off (25 min)

Everything so far used questions the corpus can answer. Production
traffic is not so polite. `UNANSWERABLE` holds four questions about
things the documentation genuinely does not cover: the TH-99 warranty,
the PK-8 wind rating, GD-200 HomeKit support, DS-12 color options.

The correct behavior on these is to say so. The base model does. The
adapter was trained on thousands of question and answer pairs where the
answer was always a confident JSON object, and it learned that lesson
too well: it produces a perfectly formatted, confidently cited, entirely
fabricated answer. The slides call this over adaptation; a support agent
would call it the answer that gets a customer's breaker panel rewired
wrong.

`abstention_report(df_base_ans, df_lora_ans, df_base_un, df_lora_un)`
takes the four result frames (answerable and unanswerable, base and
adapter) and returns a nested dict:

```python
{"base": {"abstention_rate_unanswerable": ...,
          "false_answer_rate_unanswerable": ...,
          "abstention_rate_answerable": ...},
 "lora": {...same keys...}}
```

where `false_answer_rate_unanswerable` is the fraction of unanswerable
questions that were **not** abstained (on a question with no answer in
the corpus, answering at all is a false answer), and
`abstention_rate_answerable` catches the opposite failure, refusing
questions it should answer. Plain floats.

**Worked target output.**

```python
{"base": {"abstention_rate_unanswerable": 1.0,
          "false_answer_rate_unanswerable": 0.0,
          "abstention_rate_answerable": 0.0},
 "lora": {"abstention_rate_unanswerable": 0.0,
          "false_answer_rate_unanswerable": 1.0,
          "abstention_rate_answerable": 0.0}}
```

**Done when** F1 passes.

**Group discussion (10 min).** With your table group: would you ship
this adapter to the Cordwell support team? The metrics say format went
from 0 to 100 percent and faithfulness held at 1.0 on answerable
questions, and the same metrics say every unanswerable question now gets
a confident fabrication. What would you change first: the training data,
the system prompt, or the deployment gating? Reference the abstention
slide from this morning and be ready to defend one answer.


In [ ]:
# ============ SOLUTION ============
def abstention_report(df_base_ans, df_lora_ans, df_base_un, df_lora_un):
    """Summarize abstention behavior for base and adapter."""
    def side(df_ans, df_un):
        return {
            "abstention_rate_unanswerable": float(df_un["abstained"].mean()),
            "false_answer_rate_unanswerable": float((~df_un["abstained"]).mean()),
            "abstention_rate_answerable": float(df_ans["abstained"].mean()),
        }
    return {"base": side(df_base_ans, df_base_un),
            "lora": side(df_lora_ans, df_lora_un)}

In [ ]:
UN_BASE = UN_LORA = ABSTENTION_REPORT = None
if CHAIN is None or DF_BASE is None:
    print("Complete Parts B and E first.")
else:
    try:
        set_adapter(False)
        UN_BASE = evaluate_set(CHAIN, UNANSWERABLE, JUDGE)
        set_adapter(True)
        UN_LORA = evaluate_set(CHAIN, UNANSWERABLE, JUDGE)
        print("Adapter answers to unanswerable questions:")
        for _, row in UN_LORA.iterrows():
            print(f"  Q: {row['question']}")
            print(f"  A: {row['answer']}\n")
        ABSTENTION_REPORT = abstention_report(DF_BASE, DF_LORA, UN_BASE, UN_LORA)
        print(json.dumps(ABSTENTION_REPORT, indent=2))
    except NotImplementedError:
        print("abstention_report (or an earlier function) not implemented yet.")
    finally:
        set_adapter(True)

soft("F1_abstention_tradeoff", lambda: ABSTENTION_REPORT is not None and
     ABSTENTION_REPORT["base"]["abstention_rate_unanswerable"] == 1.0 and
     ABSTENTION_REPORT["lora"]["false_answer_rate_unanswerable"] == 1.0 and
     ABSTENTION_REPORT["base"]["false_answer_rate_unanswerable"] == 0.0 and
     ABSTENTION_REPORT["lora"]["abstention_rate_answerable"] == 0.0)

## Part G. Stretch: degrade retrieval and diagnose (30 min, optional)

Every measurement so far ran with `TOP_K = 4`, and every answerable
question had its gold chunk retrieved. This stretch simulates a
retrieval regression by cutting the chain to one retrieved chunk and
diagnosing the damage with the diagnostic table's decision rule.

`run_degraded(make_retriever, llm, queries, judge)` builds a k equal to 1
retriever and fresh chain, evaluates the answerable set with the adapter
enabled, and returns the results frame plus a diagnosis dict with
`context_hit_rate`, `format_rate`, `mean_faithfulness`, and a
`diagnosis` string chosen by the table's rule: below 1.0 context hit
means fix retrieval before touching generation.

**Instructor reveal, for the debrief.** The degraded frame contains two
different failure shapes, engineered by the corpus decoys:

* **Fabrication.** The circuit breaker question now retrieves only the
  quickstart chunk, which mentions the breaker panel but not the sizing.
  The adapter fabricates from what it has: it answers "wait 7 minutes
  before installing your thermostat" with the quickstart cited,
  faithfulness 0.0. Retrieval missed, generation invented, the metric
  caught it.
* **Faithful but useless.** The return window question now retrieves
  only the returns portal chunk, which truthfully describes how to print
  a shipping label and says nothing about the 90 day window. The adapter
  quotes it verbatim: faithfulness 1.0, context hit False, and the
  customer's actual question is unanswered. This is the diagnostic
  table's top row, high faithfulness with low recall, and it is the
  failure that looks healthy on a faithfulness dashboard. Only the
  retrieval metric exposes it.

Format stays at 1.0 throughout: the adapter's formatting is robust to
retrieval quality, which is exactly why format adherence alone is not an
evaluation.


In [ ]:
# ============ SOLUTION ============
def run_degraded(make_retriever, llm, queries, judge):
    """Stretch: evaluate the chain with retrieval cut to one chunk."""
    set_adapter(True)
    chain_k1 = build_chain(llm, make_retriever(1))
    df = evaluate_set(chain_k1, queries, judge)
    faith = df["faithfulness"].dropna()
    hit = float(df["context_hit"].mean())
    mean_faith = float(faith.mean()) if len(faith) else None
    if hit < 1.0:
        diagnosis = "fix_retrieval"
    elif mean_faith is not None and mean_faith < 1.0:
        diagnosis = "fix_generation"
    else:
        diagnosis = "healthy"
    return df, {"context_hit_rate": hit, "format_rate": float(df["format_valid"].mean()),
                "mean_faithfulness": mean_faith, "diagnosis": diagnosis}

In [ ]:
DF_K1 = DIAGNOSIS = None
if CHAIN is None:
    print("Complete Part B first.")
else:
    try:
        DF_K1, DIAGNOSIS = run_degraded(BACKEND.make_retriever, LLM, ANSWERABLE, JUDGE)
        print(DF_K1[["format_valid", "faithfulness", "context_hit"]].to_string())
        print()
        for _, row in DF_K1[~DF_K1["context_hit"]].iterrows():
            print(f"MISS  Q: {row['question']}")
            print(f"      A: {row['answer']}")
            print(f"      retrieved: {row['sources']}  faithfulness: {row['faithfulness']}\n")
        print(json.dumps(DIAGNOSIS, indent=2))
    except NotImplementedError:
        print("Stretch not attempted. That is fine, it is optional.")

soft("G1_degraded_diagnosis", lambda: DIAGNOSIS is not None and
     abs(DIAGNOSIS["context_hit_rate"] - 4 / 6) < 1e-9 and
     DIAGNOSIS["format_rate"] == 1.0 and
     DIAGNOSIS["mean_faithfulness"] < 1.0 and
     DIAGNOSIS["diagnosis"] == "fix_retrieval")

## Scoreboard

Re-run this cell any time. All 22 passing with the offline backend means
the lab is complete; the stretch counts as the 22nd.


In [ ]:
parts = {}
for name, ok in RESULTS.items():
    parts.setdefault(name[0], []).append(ok)
for p in sorted(parts):
    got = sum(parts[p]); tot = len(parts[p])
    print(f"Part {p}: {got}/{tot}")
score()